# Human Activity Recognition (HAR): Hyperparameter Tuning

This notebook reproduces the tuning of the LdaBoost-based approach

In [1]:
# Reproducible setup and imports
import os
import random
import logging
from typing import Dict

import numpy as np
import pandas as pd

import optuna
from optuna.samplers import TPESampler

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Import LdaBoost from local package
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset and preprocessing


In [2]:
import re

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in df.columns]
    return df

train_df = pd.read_csv("Data/train.csv")
test_df = pd.read_csv("Data/test.csv", names=list(train_df.columns), header=0)

train_df = clean_column_names(train_df)
test_df = clean_column_names(test_df)

X = train_df.drop(columns="Activity")
y = train_df["Activity"]
X_test = test_df.drop(columns="Activity")
y_test = test_df["Activity"]

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_test_enc = le.transform(y_test)

X_np = X.to_numpy()



## Optuna study for LdaBoost


In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 1200, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "random_state": RANDOM_SEED,
    }

    # Cross-validation using StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr_idx, te_idx in cv.split(X_np, y_enc):
        X_tr, X_te = X_np[tr_idx], X_np[te_idx]
        y_tr, y_te = y_enc[tr_idx], y_enc[te_idx]

        model = LdaBoost(**params)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        scores.append(accuracy_score(y_te, y_pred))
    return float(np.mean(scores))

sampler = TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=3)

best_params: Dict[str, object] = study.best_params
best_value = study.best_value
best_params, best_value


## Train final LdaBoost and report CV performance


In [ ]:
final_model = LdaBoost(
    n_estimators=int(best_params.get("n_estimators", 1150)),
    learning_rate=float(best_params.get("learning_rate", 0.27871532239639646)),
    max_depth=int(best_params.get("max_depth", 2)),
    random_state=RANDOM_SEED,
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
accs, f1s = [], []
for tr_idx, te_idx in cv.split(X_np, y_enc):
    X_tr, X_te = X_np[tr_idx], X_np[te_idx]
    y_tr, y_te = y_enc[tr_idx], y_enc[te_idx]

    m = LdaBoost(
        n_estimators=final_model.n_estimators,
        learning_rate=final_model.learning_rate,
        max_depth=final_model.max_depth,
        random_state=RANDOM_SEED,
    )
    m.fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    accs.append(accuracy_score(y_te, y_pred))
    f1s.append(f1_score(y_te, y_pred, average="macro"))

print(f"Final LdaBoost — Accuracy: mean={np.mean(accs):.4f} std={np.std(accs):.4f}")
print(f"Final LdaBoost — F1 macro: mean={np.mean(f1s):.4f} std={np.std(f1s):.4f}")
